# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display summary from the metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will list all record sets in the dataset, as well as the fields and columns available in each record set. All references are made via their `@id` attributes.

In [ ]:
# List all record sets by @id
print('Record sets in the dataset:')
for record_set in metadata.recordSets:
    print(f"- RecordSet name: {record_set.name}, @id: {record_set.id}")

    print('  Fields:')
    if hasattr(record_set, 'fields'):
        for field in record_set.fields:
            print(f"    - Field: {field.name}, @id: {field.id}, dataType: {getattr(field, 'dataType', None)}")
    print('  Columns (from source files):')
    if hasattr(record_set, 'columns'):
        for column in record_set.columns:
            print(f"    - Column name: {column.name}, @id: {column.id}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract all record sets for generality, but select one for further analysis and demonstration.

In [ ]:
# Extract data from each record set
record_sets = [record_set.id for record_set in metadata.recordSets]
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records) > 0:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records from record set {record_set_id}")
        else:
            print(f"No records found in record set {record_set_id}")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

# For this dataset, let's select the first non-empty record set
selected_record_set_id = None
for rec_id, df in dataframes.items():
    if not df.empty:
        selected_record_set_id = rec_id
        break

if selected_record_set_id is not None:
    print(f"\nSelected record set for demonstration: {selected_record_set_id}")
    print("Columns:", dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We'll demonstrate these steps for a numeric field, referencing everything by `@id` or original column names.

In [ ]:
import numpy as np

if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]

    # Try to automatically find a numeric column for demonstration
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    if numeric_field is not None:
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        threshold = round(threshold, 2)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to find a categorical (string) field for grouping
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break

        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("Could not find a numeric field in the selected record set for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We show a histogram for the identified numeric field and a bar plot for group statistics if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if 'group_field' in locals() and group_field is not None:
        plt.figure(figsize=(10,4))
        order = grouped_df.sort_values(numeric_field, ascending=False)[group_field].tolist()
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field, order=order)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
This notebook demonstrated how to use the Croissant schema and the `mlcroissant` library to programmatically explore, extract, and analyze a scientific dataset using only its semantic metadata and interoperable record set/field `@id` references.

Key steps included:
- Dynamic listing of record sets, fields, and columns via metadata.
- Data extraction using record set `@id`s.
- Basic filtering, normalization, and grouping based on semantic field identifiers.
- Simple visualizations of selected numeric data.

Further analysis can take advantage of the structured, machine-readable schema for robust and repeatable workflows.